In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)
df = df.sort_values("Time").reset_index(drop=True)

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
train_end = int(len(df) * 0.60)
validation_end = int(len(df) * 0.80)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

print(f"Training rows:   {len(train_df):,}")
print(f"Validation rows: {len(validation_df):,}")
print(f"Test rows:       {len(test_df):,}")

Training rows:   170,884
Validation rows: 56,961
Test rows:       56,962


In [ ]:
def summarize_split(name, split_df):
    fraud_count = split_df["Class"].sum()
    fraud_rate = split_df["Class"].mean()

    return {
        "split": name,
        "transactions": len(split_df),
        "fraud_count": int(fraud_count),
        "fraud_rate": fraud_rate,
        "start_hour": split_df["Time"].min() / 3600,
        "end_hour": split_df["Time"].max() / 3600,
    }


split_summary = pd.DataFrame(
    [
        summarize_split("Train", train_df),
        summarize_split("Validation", validation_df),
        summarize_split("Test", test_df),
    ]
)

split_summary

In [ ]:
display_summary = split_summary.copy()
display_summary["fraud_rate"] = (
    display_summary["fraud_rate"] * 100
).map("{:.4f}%".format)

display_summary

### Here, we use a chronological split strategy rather than mixing transactions over time.  This better represents a model that trains from earlier transactions and applies gains to later transactions.

## This datasets covers approximnately two days and will not represent long term concept drift or changing fraud strategies.  We will later use a stratified random split as a sensitivity analysis

In [ ]:
FEATURES = [column for column in df.columns if column != "Class"]
TARGET = "Class"

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_validation = validation_df[FEATURES]
y_validation = validation_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print(X_train.shape, y_train.shape)
print(X_validation.shape, y_validation.shape)
print(X_test.shape, y_test.shape)

In [ ]:
def evaluate_scores(y_true, scores, model_name):
    return {
        "model": model_name,
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }

In [ ]:
#establlish a no-skill baseline assigning each transaction the same score equal to the mean of the training labels. This is equivalent to predicting the probability of fraud for each transaction as the overall fraud rate in the training set.
no_skill_scores = np.full(
    shape=len(y_validation),
    fill_value=y_train.mean(),
)

baseline_results = evaluate_scores(
    y_validation,
    no_skill_scores,
    "No-skill baseline",
)

baseline_results

In [ ]:
all_legitimate_predictions = np.zeros(len(y_validation), dtype=int)

naive_accuracy = (
    all_legitimate_predictions == y_validation.to_numpy()
).mean()

naive_recall = 0.0

print(f"All-legitimate accuracy: {naive_accuracy:.4%}")
print(f"All-legitimate fraud recall: {naive_recall:.1%}")

Although the all-legitimate classifier achieves very high accuracy, it detects
no fraudulent transactions. Average precision and fraud-class recall are therefore
more informative than accuracy for this dataset.

In [ ]:
scale_features = ["Time", "Amount"]
pca_features = [f"V{i}" for i in range(1, 29)]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), scale_features),
        ("pca", "passthrough", pca_features),
    ]
)

In [ ]:
logistic_unweighted = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_unweighted.fit(X_train, y_train)

unweighted_scores = logistic_unweighted.predict_proba(
    X_validation
)[:, 1]

In [ ]:
unweighted_results = evaluate_scores(
    y_validation,
    unweighted_scores,
    "Logistic regression",
)

unweighted_results

In [ ]:
logistic_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_balanced.fit(X_train, y_train)

balanced_scores = logistic_balanced.predict_proba(
    X_validation
)[:, 1]

In [ ]:
balanced_results = evaluate_scores(
    y_validation,
    balanced_scores,
    "Class-weighted logistic regression",
)

balanced_results

In [ ]:
results = pd.DataFrame(
    [
        baseline_results,
        unweighted_results,
        balanced_results,
    ]
).sort_values("average_precision", ascending=False)

results

In [ ]:
def plot_precision_recall(y_true, scores, label):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)

    plt.plot(
        recall,
        precision,
        label=f"{label} (AP={ap:.3f})",
    )


plt.figure(figsize=(9, 6))

plot_precision_recall(
    y_validation,
    unweighted_scores,
    "Logistic regression",
)

plot_precision_recall(
    y_validation,
    balanced_scores,
    "Class-weighted logistic regression",
)

plt.axhline(
    y=y_validation.mean(),
    color="gray",
    linestyle="--",
    label=f"No-skill baseline ({y_validation.mean():.4f})",
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation Precision–Recall Curves")
plt.legend()
plt.show()

In [ ]:
default_predictions = (balanced_scores >= 0.5).astype(int)

print(
    classification_report(
        y_validation,
        default_predictions,
        target_names=["Legitimate", "Fraud"],
        digits=4,
    )
)

In [ ]:
cm = confusion_matrix(y_validation, default_predictions)

cm_df = pd.DataFrame(
    cm,
    index=["Actual legitimate", "Actual fraud"],
    columns=["Predicted legitimate", "Predicted fraud"],
)

cm_df